DATA WRANGLING

In [65]:
import numpy as np
import pandas as pd

In [66]:
# Load the dataset
db = pd.read_csv('Comune-di-Milano-Esercizi-di-vicinato-in-sede-fissa.csv',sep=';',encoding='unicode_escape')

In [67]:
# Rename columns and convert data types
db = db.rename(columns ={"Tipo via":"Tipo Via","Codice via": "Codice Via"})
db["Codice Via"] = pd.to_numeric(db["Codice Via"], errors="coerce").astype("Int64")
db["ZD"] = pd.to_numeric(db["ZD"], errors="coerce").astype("Int64")
db.dtypes

Settore Merceologico            object
Insegna                         object
Ubicazione                      object
Tipo Via                        object
Via                             object
Civico                          object
Codice Via                       Int64
ZD                               Int64
Settore Storico Cf Preval       object
Superficie Vendita             float64
Superficie Altri Usi           float64
Superficie Tabelle Speciali    float64
Superficie Totale              float64
dtype: object

In [ ]:
# Convert all string values to lowercase
db = db.applymap(lambda x: x.lower() if isinstance(x, str) else x)

C:\Users\Utente\AppData\Local\Temp\ipykernel_20492\568537625.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  db = db.applymap(lambda x: x.lower() if isinstance(x, str) else x)


In [69]:
# Get unique values in 'Settore Merceologico' column
db['Settore Merceologico'].unique()

array([nan, 'alimentare', 'alimentare;', 'alimentare;alimentare',
       'alimentare;non alimentare',
       'alimentare;non alimentare;tabella speciale carburanti',
       'alimentare;non alimentare;tabella speciale carburanti;tabella speciale monopolio',
       'alimentare;non alimentare;tabella speciale farmacie',
       'alimentare;non alimentare;tabella speciale monopolio',
       'alimentare;non alimentare;tabella speciale monopolio;tabella speciale carburanti',
       'alimentare;tabella speciale carburanti',
       'alimentare;tabella speciale farmacie',
       'alimentare;tabella speciale monopolio', 'non alimentare',
       'foglio 216 part. 83 sub. 716 (z.d. 8)', ' automobili',
       'non alimentare;alimentare',
       'non alimentare;alimentare;tabella speciale carburanti',
       'non alimentare;alimentare;tabella speciale monopolio',
       'non alimentare;tabella speciale carburanti',
       'non alimentare;tabella speciale carburanti;alimentare',
       'non alimentare

In [70]:
# Get number of unique values in 'Settore Merceologico' column
db['Settore Merceologico'].nunique()

33

In [71]:
# Split 'Settore Merceologico' into multiple boolean columns
cats = {
    "Alimentare": "alimentare",
    "Non Alimentare": "non alimentare",
    "Tabella Speciale Carburanti": "tabella speciale carburanti",
    "Tabella Speciale Farmacie": "tabella speciale farmacie",
    "Tabella Speciale Monopolio": "tabella speciale monopolio",
}

allowed = set(cats.values())

def split_settore(value):
    # If missing -> all NaN
    if pd.isna(value):
        return pd.Series({k: np.nan for k in cats})

    # Split and clean parts
    parts = [p.strip().lower() for p in str(value).split(";") if p.strip()]

    # If any part is not allowed -> all NaN
    if any(p not in allowed for p in parts) or not parts:
        return pd.Series({k: np.nan for k in cats})

    # Build output
    out = {}
    for col, key in cats.items():
        out[col] = key in parts

    return pd.Series(out)

# Apply the function to create new columns
db[list(cats)] = (
    db["Settore Merceologico"]
    .str.lower()
    .apply(split_settore)
    .astype("boolean")
)

In [72]:
# Drop the original 'Settore Merceologico' column and reorder columns
db = db.drop(['Settore Merceologico'], axis=1)
db = db[['Alimentare', 'Non Alimentare', 'Tabella Speciale Carburanti', 
         'Tabella Speciale Farmacie', 'Tabella Speciale Monopolio', 'Insegna', 
         'Ubicazione', 'Tipo Via', 'Via', 'Civico', 'Codice Via', 'ZD', 'Settore Storico Cf Preval',
         'Superficie Vendita', 'Superficie Altri Usi',
         'Superficie Tabelle Speciali', 'Superficie Totale']]

In [ ]:
# Address processing

# Create a copy of relevant columns
db2 = db[["Ubicazione", "Tipo Via", "Via", "Civico"]].copy(deep=True)

# Cleaned base address (Tipo Via + Via)
base = (
    db2["Tipo Via"].fillna("").astype(str).str.strip() + " " +
    db2["Via"].fillna("").astype(str).str.strip()
)

# Collapse multiple spaces and trim
base = base.str.replace(r"\s+", " ", regex=True).str.strip()

# Cleaned Civico
civico = db2["Civico"].fillna("").astype(str).str.strip()

# Construct full Indirizzo
db2.loc[:, "Indirizzo"] = np.where(
    civico == "",
    base,
    (base + " n. " + civico)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
)

# Check if Indirizzo is in Ubicazione
db2.loc[:, "Match Ubicazione"] = db2.apply(
    lambda r: r["Indirizzo"] in str(r["Ubicazione"]) if r["Indirizzo"] else False,
    axis=1
)

# Extract Resto Ubicazione
# Assuming Tipo Via, Via, and Civico from the original data are correct
db2.loc[:, "Resto Ubicazione"] = db2.apply(
    lambda r: (
        str(r["Ubicazione"]).replace(r["Indirizzo"], "").strip()
        if r["Match Ubicazione"]
        and isinstance(r["Ubicazione"], str)
        and isinstance(r["Indirizzo"], str)
        else np.nan
    ),
    axis=1
)

# Clean Resto Ubicazione
db2["Resto Ubicazione"] = (
    db2["Resto Ubicazione"]
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"^\s*;+\s*", "", regex=True)
        .str.strip()
)

In [74]:
# Drop intermediate columns
db2 = db2.drop(['Ubicazione', 'Indirizzo', 'Match Ubicazione'], axis=1)

In [75]:
# Split Resto Ubicazione to obtain Accesso and Isolato columns
tmp = db2["Resto Ubicazione"].str.split(";", expand=True)
tmp = tmp.apply(lambda col: col.str.strip().str.lower())

def reorganize(row):
    # Valid string values
    vals = [x for x in row if isinstance(x, str) and x.strip() != ""]

    # Categories
    isolato_parts = [x for x in vals if x.startswith("isolato:")]
    accesso_parts = [x for x in vals if x.startswith("accesso:")]

    # Isolato value (nan if not found)
    if isolato_parts:
        isolato_val = isolato_parts[0].replace("isolato:", "").strip()
    else:
        isolato_val = np.nan

    # Accesso value (nan if not found)
    if accesso_parts:
        accesso_val = (
            accesso_parts[0]
            .replace("accesso: accesso", "")
            .replace("accesso:", "")
            .strip()
        )
    else:
        accesso_val = np.nan

    # Resto Ubicazione (remaining parts)
    used = set(isolato_parts + accesso_parts)
    resto_list = [x for x in vals if x not in used]

    # Remaining Resto Ubicazione
    if resto_list:
        resto_val = "; ".join(resto_list).strip()
    else:
        resto_val = np.nan

    return pd.Series([isolato_val, accesso_val, resto_val])

# Apply reorganize to tmp
new_cols = tmp.apply(reorganize, axis=1)
new_cols.columns = ["Isolato", "Accesso", "Resto Ubicazione Clean"]

# Attach new columns to db2
db2[["Isolato", "Accesso"]] = new_cols[["Isolato", "Accesso"]]
db2["Resto Ubicazione"] = new_cols["Resto Ubicazione Clean"]

db2.to_csv('diq_dataw_processed.csv', sep=';', index=False)

In [ ]:
# Clean Resto Ubicazione for (z.d. ...) entries
tmp = db2["Resto Ubicazione"].str.split(";", expand=True)
tmp = tmp.apply(lambda col: col.str.strip().str.lower())

def reorganize(row):
    # Valid string values
    vals = [x for x in row if isinstance(x, str) and x.strip() != ""]

    # z.d. parts
    zd_parts = [x for x in vals if x.startswith("(z.d. ")]

    # Resto Ubicazione (remaining parts)
    used = set(zd_parts)
    resto_list = [x for x in vals if x not in used]

    # Remaining Resto Ubicazione
    if resto_list:
        resto_val = "; ".join(resto_list).strip()
    else:
        resto_val = np.nan

    return pd.Series([resto_val])

# Apply reorganize to tmp
new_cols = tmp.apply(reorganize, axis=1)
new_cols.columns = ["Resto Ubicazione Clean"]

# Attach new cleaned Resto Ubicazione to db2
db2["Resto Ubicazione"] = new_cols["Resto Ubicazione Clean"]

In [77]:
# Count all null values for each column
db2.isnull().sum()

Tipo Via                2
Via                     1
Civico                650
Resto Ubicazione    19591
Isolato              4775
Accesso              7843
dtype: int64

In [78]:
# Dropping Ubicazione in the original db after extracting data
db = db.drop(['Ubicazione'], axis=1)
db[["Isolato","Accesso","Resto Ubicazione"]] = db2[["Isolato","Accesso","Resto Ubicazione"]]

# Rearranging columns
db = db[['Alimentare', 'Non Alimentare', 'Tabella Speciale Carburanti', 
         'Tabella Speciale Farmacie', 'Tabella Speciale Monopolio', 'Insegna', 
         'Tipo Via', 'Via', 'Civico', 'Codice Via', 'ZD', 'Isolato','Accesso','Resto Ubicazione', 'Settore Storico Cf Preval',
         'Superficie Vendita', 'Superficie Altri Usi', 'Superficie Tabelle Speciali', 'Superficie Totale']]

In [ ]:
# Save the wrangled dataset
db.to_csv("db_wrangled.csv", index=False)